# 2048 teacher -> linear student (Colab)

Runs the **teacher** (a depth-limited expectimax search with a hand-written heuristic)
on many 2048 games, labels every position with

* the move the teacher played (**policy label** -> multinomial logistic regression), and
* how many moves were left before a 2048 tile appeared (**value label** -> ridge regression),

then writes a single `model.json` you drop into the web app.

**Setup:** nothing to do. The first cell clones this repo into
`/content` (or uses the folder you are already in). Runtime -> GPU
recommended (T4 is enough).


In [ ]:
# --- setup -------------------------------------------------------------------
REPO_URL = "https://github.com/swaftmc/2048-ai.git"
FOLDER = "2048-ai"        # folder name inside /content

import os, sys, pathlib, shutil

try:
    import subprocess
    subprocess.run(["pip", "install", "-q", "torch", "numpy", "scikit-learn"], check=False)
except Exception as exc:
    print("pip install skipped:", exc)

here = pathlib.Path.cwd()
if (here / "src").exists():
    root = here                          # already inside the project (local run)
else:
    content = pathlib.Path("/content")
    if not content.exists():
        content = here
    root = content / FOLDER
    if REPO_URL and not (root / "src").exists():
        shutil.rmtree(root, ignore_errors=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    if not (root / "src").exists():
        raise SystemExit(
            f"Cannot find {root / 'src'}. In Colab this cell clones {REPO_URL}; "
            "locally, run the notebook from inside the 2048-ai folder.")
sys.path.insert(0, str(root))
os.chdir(root)

import torch
print("root:", root)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


In [ ]:
# --- config ------------------------------------------------------------------
# depth         : expectimax depth. Measured on CPU: depth 2 -> 2048 in 17-50% of
#                 games, depth 3 -> 75%. Depth 4 is the quality setting but costs
#                 ~8x the leaves of depth 3 with the settings below.
# cell_samples  : empty cells enumerated per chance node (16 = exact, 8 = sampled)
#                 leaves per move per game = (8 * cell_samples) ** depth
# batch         : games played in parallel by the vectorised teacher. Keep
#                 batch * leaves * 64 bytes under ~4 GB of VRAM.
# games         : teacher games to play. A game is ~1000 moves, so budget
#                 games * 1000 / batch * seconds-per-batch-move.
#                 Run the smoke cell above first to see seconds-per-move on your GPU.
GAMES       = 200
DEPTH       = 3
CELL_SAMPLES = 8
BATCH       = 8           # games played in parallel
EVAL_GAMES  = 20          # student games per evaluation mode
SEED        = 0
OUT         = "model.json"
print(f"{GAMES} teacher games, depth {DEPTH}, {CELL_SAMPLES} cells/node, batch {BATCH}")
print(f"leaves per move per game: {(8 * CELL_SAMPLES) ** DEPTH:,}")

In [ ]:
# --- smoke test: one teacher move (catches device / VRAM problems in seconds) ----
import time
import torch
from src import teacher

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device,
      torch.cuda.get_device_name(0) if device == "cuda" else "")
gen = torch.Generator(device=device)
gen.manual_seed(0)
boards = torch.zeros((min(BATCH, 4), 4, 4), dtype=torch.int32, device=device)
boards = teacher.spawn_boards(boards, gen)
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
vals, moves, _ = teacher.expectimax(boards, DEPTH, CELL_SAMPLES, gen, teacher.WEIGHTS)
if device == "cuda":
    torch.cuda.synchronize()
print(f"one move at depth {DEPTH}: {time.time() - t0:.1f}s, moves {moves.tolist()}")
if device == "cuda":
    print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 2 ** 30:.2f} GiB")


In [ ]:
# --- run --------------------------------------------------------------------
from src.pipeline import run

metrics = run(games=GAMES, depth=DEPTH, cell_samples=CELL_SAMPLES, batch=BATCH,
              seed=SEED, out=OUT, eval_games=EVAL_GAMES)
print("\nteacher:", metrics["teacher"])
print("policy :", metrics["policy"])
print("value  :", metrics["value"])

In [ ]:
# --- get the model out -------------------------------------------------------
import json, shutil
from google.colab import files

try:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(OUT, "/content/drive/MyDrive/2048_model.json")
    print("saved to MyDrive/2048_model.json")
except Exception as exc:
    print("Drive not available:", exc)
files.download(OUT)